# 15장 STT와 TTS 보조 실습

이 장은 PDF 교재의 LLM 챗봇, Gradio, 멀티모달 보조 실습 흐름을 바탕으로 음성 인터페이스를 구성하는 방법을 정리합니다.

STT는 Speech-to-Text의 약자로 음성을 텍스트로 변환하는 기술입니다. TTS는 Text-to-Speech의 약자로 텍스트를 음성으로 변환하는 기술입니다. LLM 챗봇에 STT와 TTS를 연결하면 사용자가 말로 질문하고, 챗봇이 음성으로 답하는 형태의 애플리케이션을 만들 수 있습니다.

## 이 장에서 다루는 내용

1. STT와 TTS 개념
2. LLM 챗봇에서 음성 인터페이스가 필요한 이유
3. 음성 처리 전체 흐름
4. 설치 준비
5. 오디오 파일 다루기
6. Whisper 기반 STT 개념 예시
7. gTTS 기반 TTS 개념 예시
8. LLM 챗봇과 STT 연결
9. LLM 챗봇과 TTS 연결
10. Gradio 음성 챗봇 UI
11. 오류 해결과 실습 팁

이 장은 `LLM/llm.ipynb`의 Gradio 챗봇 흐름을 음성 입출력으로 확장하는 보조 실습입니다.


## 15.1 STT와 TTS 개념

STT와 TTS는 LLM 자체의 핵심 구조는 아니지만, LLM 애플리케이션의 사용자 경험을 확장하는 데 자주 사용됩니다.

| 구분 | 전체 이름 | 역할 | 예시 |
|---|---|---|---|
| STT | Speech-to-Text | 음성을 텍스트로 변환 | 사용자의 말소리를 질문 문장으로 변환 |
| TTS | Text-to-Speech | 텍스트를 음성으로 변환 | 챗봇 답변을 음성 파일로 변환 |

LLM 챗봇에 연결하면 다음 흐름이 됩니다.

```text
사용자 음성
  -> STT
  -> 텍스트 질문
  -> LLM
  -> 텍스트 답변
  -> TTS
  -> 음성 답변
```

따라서 STT와 TTS는 텍스트 기반 LLM 챗봇의 앞뒤에 붙는 입출력 변환 계층이라고 볼 수 있습니다.


## 15.2 음성 인터페이스가 필요한 이유

음성 인터페이스는 다음 상황에서 유용합니다.

- 키보드 입력이 불편한 환경
- 시각적 UI보다 대화형 사용성이 중요한 서비스
- 상담, 안내, 교육, 접근성 기능
- 스마트 스피커나 모바일 앱 형태의 챗봇
- 시연용 LLM 앱에서 몰입감 있는 사용자 경험 제공

다만 음성 인터페이스는 텍스트 챗봇보다 고려할 점이 더 많습니다.

- 주변 소음 때문에 STT 정확도가 낮아질 수 있습니다.
- 발음, 억양, 언어 설정에 따라 인식 결과가 달라질 수 있습니다.
- 긴 답변은 TTS로 들을 때 피로도가 높을 수 있습니다.
- 음성 파일 저장 시 개인정보와 민감 정보 관리가 필요합니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - 설치 준비
# 이 셀은 음성 처리 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# gradio는 음성 입력과 음성 출력이 있는 웹 UI를 만들 때 사용합니다.
!pip install gradio

# gTTS는 Google Text-to-Speech를 사용해 텍스트를 음성 파일로 변환합니다.
!pip install gTTS

# pydub은 오디오 파일 형식 변환과 간단한 오디오 처리를 할 때 사용합니다.
!pip install pydub

# openai-whisper는 Whisper 기반 음성 인식 모델을 로컬에서 사용할 때 쓰는 패키지입니다.
!pip install openai-whisper

# langchain과 langchain-ollama는 LLM 챗봇 답변 생성을 위해 사용합니다.
!pip install langchain langchain-ollama


## 15.3 오디오 파일 다루기

Jupyter Notebook과 Gradio에서 음성을 다룰 때는 보통 파일 경로를 사용합니다.

Gradio의 `Audio` 컴포넌트는 녹음된 음성을 파일 경로 또는 numpy 배열 형태로 함수에 전달할 수 있습니다. 실습에서는 이해하기 쉬운 파일 경로 방식을 기준으로 설명합니다.

음성 파일에서 확인할 주요 요소는 다음과 같습니다.

| 요소 | 의미 |
|---|---|
| sample rate | 1초에 오디오를 몇 번 샘플링했는지 나타내는 값 |
| channel | mono 또는 stereo 같은 채널 수 |
| format | wav, mp3, m4a 같은 파일 형식 |
| duration | 음성 길이 |

STT 모델은 입력 형식에 민감할 수 있으므로, 필요하면 wav 형식으로 변환해서 사용하는 것이 안정적입니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - 오디오 파일 경로 확인
# 이 셀은 오디오 파일 경로가 존재하는지 확인하는 기본 예시입니다.

# os 모듈은 파일 존재 여부와 경로 처리를 위해 사용합니다.
import os

# Path는 운영체제에 맞는 파일 경로를 객체로 다룰 때 사용합니다.
from pathlib import Path

# audio_path 변수에는 실습에 사용할 음성 파일 경로를 지정합니다.
audio_path = Path("sample_audio.wav")

# exists 메서드는 해당 경로에 파일이 실제로 있는지 확인합니다.
audio_exists = audio_path.exists()

# 파일 존재 여부를 출력합니다.
print("오디오 파일 존재 여부:", audio_exists)

# 파일이 없을 경우 사용자가 준비해야 할 파일명을 안내합니다.
if not audio_exists:
    # sample_audio.wav 파일을 현재 노트북 폴더에 두면 이후 예제를 사용할 수 있습니다.
    print("현재 폴더에 sample_audio.wav 파일을 준비하면 STT 예제를 실행할 수 있습니다.")


## 15.4 Whisper 기반 STT 개념 예시

Whisper는 음성을 텍스트로 변환하는 STT 모델입니다. 오디오 파일을 입력하면 인식된 텍스트를 반환합니다.

기본 흐름은 다음과 같습니다.

```text
Whisper 모델 로드
  -> 오디오 파일 입력
  -> transcribe 실행
  -> text 추출
```

모델 크기가 커질수록 정확도는 좋아질 수 있지만 실행 속도와 메모리 사용량이 증가합니다.

| 모델 예시 | 특징 |
|---|---|
| tiny | 빠르지만 정확도가 낮을 수 있음 |
| base | 실습용으로 비교적 가벼움 |
| small | 정확도와 속도의 균형 |
| medium 이상 | 더 높은 정확도, 더 큰 리소스 필요 |


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - Whisper STT 예시
# 이 셀은 Whisper를 사용해 음성 파일을 텍스트로 변환하는 예시입니다.

# whisper 패키지는 음성 인식 모델을 로드하고 transcribe 기능을 제공합니다.
import whisper

# whisper.load_model은 사용할 Whisper 모델을 메모리에 로드합니다.
stt_model = whisper.load_model("base")

# transcribe_audio 함수는 오디오 파일 경로를 받아 인식된 텍스트를 반환합니다.
def transcribe_audio(file_path: str) -> str:
    # 파일 경로가 비어 있으면 안내 문장을 반환합니다.
    if file_path is None:
        # 음성 파일이 없을 때의 안내 메시지입니다.
        return "음성 파일이 입력되지 않았습니다."

    # Whisper 모델의 transcribe 메서드로 음성을 텍스트로 변환합니다.
    result = stt_model.transcribe(file_path, language="ko")

    # 결과 딕셔너리에서 text 값을 꺼내고 앞뒤 공백을 제거합니다.
    text = result["text"].strip()

    # 인식된 텍스트를 반환합니다.
    return text


# sample_audio.wav 파일이 있을 때만 STT 함수를 실행합니다.
if audio_path.exists():
    # 오디오 파일을 텍스트로 변환합니다.
    recognized_text = transcribe_audio(str(audio_path))

    # 인식된 텍스트를 출력합니다.
    print(recognized_text)


## 15.5 gTTS 기반 TTS 개념 예시

gTTS는 텍스트를 음성 파일로 변환하는 간단한 TTS 도구입니다. 한국어 음성도 사용할 수 있습니다.

기본 흐름은 다음과 같습니다.

```text
텍스트 입력
  -> gTTS 객체 생성
  -> mp3 파일로 저장
  -> 오디오 출력 또는 재생
```

실습에서는 챗봇의 텍스트 답변을 mp3 파일로 저장한 뒤 Gradio에서 재생할 수 있게 만듭니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - gTTS TTS 예시
# 이 셀은 텍스트를 한국어 음성 mp3 파일로 변환하는 예시입니다.

# gTTS 클래스는 텍스트를 음성으로 변환하는 객체를 만듭니다.
from gtts import gTTS

# uuid는 매번 다른 파일명을 만들기 위해 사용합니다.
import uuid

# text_to_speech 함수는 입력 텍스트를 mp3 음성 파일로 저장하고 경로를 반환합니다.
def text_to_speech(text: str) -> str:
    # 텍스트가 비어 있으면 기본 안내 문장을 사용합니다.
    if not text:
        # 음성으로 변환할 기본 문장입니다.
        text = "변환할 텍스트가 없습니다."

    # output_dir 변수에는 음성 파일을 저장할 폴더를 지정합니다.
    output_dir = Path("tts_outputs")

    # mkdir은 출력 폴더가 없을 때 새로 만듭니다.
    output_dir.mkdir(exist_ok=True)

    # uuid4는 중복되지 않는 파일명을 만들기 위한 랜덤 ID를 생성합니다.
    file_name = f"answer_{uuid.uuid4().hex}.mp3"

    # output_path는 저장할 mp3 파일의 전체 경로입니다.
    output_path = output_dir / file_name

    # gTTS 객체를 만들고 언어를 한국어로 지정합니다.
    tts = gTTS(text=text, lang="ko")

    # save 메서드는 생성된 음성을 mp3 파일로 저장합니다.
    tts.save(str(output_path))

    # 저장된 파일 경로를 문자열로 반환합니다.
    return str(output_path)


# TTS 예시에 사용할 문장을 준비합니다.
sample_text = "안녕하세요. 이것은 텍스트를 음성으로 변환하는 예시입니다."

# sample_text를 음성 파일로 변환합니다.
sample_tts_path = text_to_speech(sample_text)

# 생성된 음성 파일 경로를 출력합니다.
print(sample_tts_path)


## 15.6 LLM 챗봇과 STT 연결

STT를 LLM 챗봇에 연결하면 음성 파일에서 인식된 텍스트를 그대로 LLM 질문으로 사용할 수 있습니다.

흐름은 다음과 같습니다.

```text
오디오 파일
  -> transcribe_audio()
  -> question 텍스트
  -> LLM chain.invoke()
  -> answer 텍스트
```

이때 STT 결과가 틀릴 수 있으므로, Gradio UI에서는 인식된 텍스트를 함께 보여주는 것이 좋습니다. 사용자가 텍스트를 확인하고 수정할 수 있으면 실습과 서비스 모두에서 안정성이 높아집니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - LLM 체인 준비
# 이 셀은 STT 결과 텍스트에 답변할 LLM 체인을 구성합니다.

# ChatPromptTemplate은 LLM에 전달할 대화형 프롬프트를 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 LLM 응답을 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama 로컬 모델을 LangChain 방식으로 호출합니다.
from langchain_ollama import ChatOllama

# llm 변수에는 사용할 Ollama 모델을 지정합니다.
llm = ChatOllama(model="llama3.1")

# voice_prompt는 음성으로 입력된 질문에 답변하기 위한 프롬프트입니다.
voice_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 챗봇의 역할을 지정합니다.
    ("system", "너는 한국어로 친절하게 답변하는 LLM 실습 도우미야."),
    # human 메시지는 STT로 변환된 질문 텍스트를 받습니다.
    ("human", "사용자의 질문: {question}"),
])

# voice_chain은 프롬프트, LLM, 출력 파서를 연결한 체인입니다.
voice_chain = voice_prompt | llm | StrOutputParser()


# answer_from_audio 함수는 음성을 텍스트로 바꾼 뒤 LLM 답변을 생성합니다.
def answer_from_audio(file_path: str) -> tuple[str, str]:
    # transcribe_audio 함수를 사용해 음성 파일을 텍스트 질문으로 변환합니다.
    question = transcribe_audio(file_path)

    # LLM 체인에 질문을 전달해 답변을 생성합니다.
    answer = voice_chain.invoke({"question": question})

    # 인식된 질문과 LLM 답변을 함께 반환합니다.
    return question, answer


## 15.7 LLM 챗봇과 TTS 연결

TTS를 연결하면 LLM의 텍스트 답변을 음성 파일로 바꿀 수 있습니다.

흐름은 다음과 같습니다.

```text
텍스트 질문
  -> LLM 답변 생성
  -> text_to_speech()
  -> mp3 파일 경로 반환
```

Gradio에서는 함수가 텍스트 답변과 오디오 파일 경로를 함께 반환하도록 만들면, 화면에 답변 텍스트와 음성 재생기를 같이 표시할 수 있습니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - 텍스트 질문을 음성 답변으로 변환
# 이 셀은 텍스트 질문을 받아 LLM 답변과 TTS 음성 파일을 함께 만드는 함수입니다.

# answer_text_and_voice 함수는 질문에 대한 텍스트 답변과 음성 파일 경로를 반환합니다.
def answer_text_and_voice(question: str) -> tuple[str, str]:
    # 질문이 비어 있으면 안내 메시지를 답변으로 사용합니다.
    if not question:
        # 빈 질문에 대한 기본 답변입니다.
        answer = "질문을 입력해 주세요."
    else:
        # LLM 체인을 실행해 텍스트 답변을 생성합니다.
        answer = voice_chain.invoke({"question": question})

    # 생성된 텍스트 답변을 TTS로 변환해 mp3 파일을 만듭니다.
    audio_output_path = text_to_speech(answer)

    # 텍스트 답변과 음성 파일 경로를 함께 반환합니다.
    return answer, audio_output_path


# 예시 질문을 준비합니다.
sample_question = "LLM이 무엇인지 간단히 설명해줘."

# 예시 질문에 대한 답변과 음성 파일을 생성합니다.
sample_answer, sample_audio_answer_path = answer_text_and_voice(sample_question)

# 텍스트 답변을 출력합니다.
print(sample_answer)

# 음성 파일 경로를 출력합니다.
print(sample_audio_answer_path)


## 15.8 Gradio 음성 챗봇 UI

Gradio를 사용하면 음성 입력, 인식된 텍스트, LLM 답변, TTS 오디오를 한 화면에 구성할 수 있습니다.

UI 구성 예시는 다음과 같습니다.

```text
Audio 입력
  -> STT 결과 Textbox
  -> LLM 답변 Textbox
  -> TTS 음성 Audio 출력
```

이 구조는 PDF 교재의 Gradio 챗봇 예제를 음성 입출력 형태로 확장한 것입니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - Gradio 음성 챗봇
# 이 셀은 음성 입력을 받아 텍스트와 음성 답변을 출력하는 Gradio UI를 만듭니다.

# gradio를 gr이라는 별칭으로 불러옵니다.
import gradio as gr


# voice_chat 함수는 오디오 입력을 받아 STT, LLM, TTS를 순서대로 실행합니다.
def voice_chat(audio_file: str) -> tuple[str, str, str]:
    # 오디오 파일이 없으면 안내 메시지를 반환합니다.
    if audio_file is None:
        # 인식 텍스트, 답변 텍스트, 오디오 출력 경로를 반환합니다.
        return "", "음성을 입력해 주세요.", None

    # STT를 실행해 음성을 텍스트 질문으로 변환합니다.
    question = transcribe_audio(audio_file)

    # LLM 체인을 실행해 질문에 대한 답변을 생성합니다.
    answer = voice_chain.invoke({"question": question})

    # LLM 답변을 TTS로 변환해 음성 파일을 생성합니다.
    answer_audio_path = text_to_speech(answer)

    # 인식된 질문, 텍스트 답변, 음성 답변 파일 경로를 반환합니다.
    return question, answer, answer_audio_path


# Blocks는 여러 Gradio 컴포넌트를 자유롭게 배치할 수 있는 UI 컨테이너입니다.
with gr.Blocks() as voice_demo:
    # Markdown 컴포넌트는 화면 제목과 설명을 표시합니다.
    gr.Markdown("# STT + LLM + TTS 음성 챗봇")

    # Audio 컴포넌트는 마이크 녹음 또는 오디오 파일 입력을 받습니다.
    audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="음성 입력")

    # Textbox는 STT로 인식된 텍스트를 표시합니다.
    recognized_text_output = gr.Textbox(label="STT 인식 결과")

    # Textbox는 LLM의 텍스트 답변을 표시합니다.
    answer_text_output = gr.Textbox(label="LLM 답변")

    # Audio 컴포넌트는 TTS로 생성된 음성 답변을 재생합니다.
    answer_audio_output = gr.Audio(label="TTS 음성 답변")

    # Button 컴포넌트는 음성 챗봇 실행 버튼입니다.
    submit_button = gr.Button("음성 질문 보내기")

    # 버튼 클릭 시 voice_chat 함수를 실행하고 세 개의 출력 컴포넌트에 결과를 보냅니다.
    submit_button.click(
        # fn에는 실행할 함수를 지정합니다.
        fn=voice_chat,
        # inputs에는 함수에 전달할 입력 컴포넌트를 지정합니다.
        inputs=audio_input,
        # outputs에는 함수 반환값을 받을 출력 컴포넌트를 지정합니다.
        outputs=[recognized_text_output, answer_text_output, answer_audio_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 음성 챗봇 UI를 확인할 수 있습니다.
# voice_demo.launch()


## 15.9 텍스트 입력과 음성 출력만 사용하는 UI

마이크 입력이 불안정하거나 Whisper 설치가 부담스러울 때는 텍스트 입력과 음성 출력만 먼저 실습해도 됩니다.

이 방식은 다음 흐름입니다.

```text
텍스트 질문
  -> LLM 답변
  -> TTS 음성 출력
```

STT를 제외하고도 TTS 연결 구조를 충분히 이해할 수 있습니다.


In [ ]:
# 교재 위치: 15장 STT와 TTS 보조 실습 - 텍스트 입력 TTS 출력 UI
# 이 셀은 텍스트 질문을 받아 답변 텍스트와 음성 파일을 출력하는 Gradio UI 예시입니다.

# text_voice_demo는 텍스트 기반 음성 출력 UI입니다.
with gr.Blocks() as text_voice_demo:
    # 화면 제목을 Markdown으로 표시합니다.
    gr.Markdown("# 텍스트 질문 + TTS 답변")

    # 질문 입력용 Textbox를 만듭니다.
    question_input = gr.Textbox(label="질문 입력")

    # LLM 답변을 표시할 Textbox를 만듭니다.
    text_answer_output = gr.Textbox(label="텍스트 답변")

    # TTS 음성 파일을 재생할 Audio 컴포넌트를 만듭니다.
    voice_answer_output = gr.Audio(label="음성 답변")

    # 실행 버튼을 만듭니다.
    text_submit_button = gr.Button("질문 보내기")

    # 버튼 클릭 시 answer_text_and_voice 함수를 실행합니다.
    text_submit_button.click(
        # fn에는 텍스트 질문 처리 함수를 지정합니다.
        fn=answer_text_and_voice,
        # inputs에는 질문 입력 Textbox를 지정합니다.
        inputs=question_input,
        # outputs에는 텍스트 답변과 음성 답변 컴포넌트를 지정합니다.
        outputs=[text_answer_output, voice_answer_output],
    )

# launch를 실행하면 로컬 웹 브라우저에서 텍스트 입력 TTS UI를 확인할 수 있습니다.
# text_voice_demo.launch()


## 15.10 실습 시 주의할 점

STT와 TTS 실습은 LLM 실습보다 외부 환경의 영향을 많이 받습니다.

확인할 사항은 다음과 같습니다.

- 마이크 권한이 브라우저에서 허용되어 있는지 확인합니다.
- Whisper 모델 로딩에는 시간이 걸릴 수 있습니다.
- CPU 환경에서는 STT 속도가 느릴 수 있습니다.
- gTTS는 인터넷 연결이 필요할 수 있습니다.
- 긴 답변은 TTS 변환 시간이 길어질 수 있습니다.
- 오디오 파일 경로에 한글이나 공백이 있을 때 일부 라이브러리에서 문제가 생길 수 있습니다.
- TTS 출력 파일이 계속 쌓이므로 주기적으로 정리하는 것이 좋습니다.
- 음성 파일에는 개인정보가 포함될 수 있으므로 저장 위치와 공유 범위를 주의합니다.


## 15.11 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| Whisper import 오류 | 패키지 미설치 | `pip install openai-whisper` 실행 |
| ffmpeg 관련 오류 | 오디오 변환 도구 미설치 | ffmpeg 설치 후 PATH 확인 |
| STT 결과가 부정확함 | 소음, 작은 목소리, 언어 설정 문제 | 조용한 환경에서 녹음하고 language='ko' 지정 |
| gTTS 오류 | 네트워크 문제 또는 요청 실패 | 인터넷 연결 확인 후 다시 실행 |
| Gradio 마이크 입력 실패 | 브라우저 권한 문제 | 마이크 권한 허용 또는 파일 업로드 사용 |
| Ollama 호출 실패 | Ollama 서버 미실행 | Ollama 실행 후 모델 이름 확인 |
| 음성 파일 재생 안 됨 | 파일 경로 또는 형식 문제 | mp3 파일 생성 여부와 경로 확인 |


## 15.12 정리

이 장에서는 STT와 TTS를 사용해 LLM 챗봇을 음성 인터페이스로 확장하는 방법을 정리했습니다.

핵심 정리는 다음과 같습니다.

- STT는 음성을 텍스트로 바꾸는 기술입니다.
- TTS는 텍스트를 음성으로 바꾸는 기술입니다.
- LLM 챗봇의 앞에는 STT를, 뒤에는 TTS를 연결할 수 있습니다.
- Whisper는 음성 인식 실습에 사용할 수 있습니다.
- gTTS는 텍스트를 mp3 음성 파일로 변환하는 간단한 도구입니다.
- Gradio를 사용하면 음성 입력, 텍스트 답변, 음성 출력을 한 화면에 구성할 수 있습니다.
- 음성 파일에는 개인정보가 들어갈 수 있으므로 저장과 공유에 주의해야 합니다.

다음 장에서는 이미지 분류와 멀티모달 실습을 다룹니다. 텍스트뿐 아니라 이미지 데이터를 모델에 입력하거나 이미지 분류 모델을 사용하는 방식으로 LLM 실습 범위를 넓혀 봅니다.
